# Fault-labelled ML training data

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mosesfalowo/ogbonlab-notebooks/blob/main/notebooks/ssm_fault_ml_dataset.ipynb)


Generated by [OgbonLab](https://ogbonlab.com), Synthetic Seismic Modeling course.

**Assumptions**: Course defaults. Edit the parameters cell and run again; every value is yours to change.

**Workflow**:
1. 3.5 Fault-Detection Training Pairs
2. 10.1 Capstone: Fault-Detection Dataset
3. 9.5 Export Your Model

In [ ]:
# Ask Ogbon starter: Generate ML training data
# Generated by OgbonLab (ogbonlab.com), Synthetic Seismic Modeling course.
import numpy as np
import matplotlib.pyplot as plt

# --- parameters from your interview ---
N_SAMPLES = 500     # interview asked for 2000; raise this for the full run
NX = NZ = 128
F_HZ = 28.0
MAX_THROW = 14
SEED = 7
LABEL = 'fault'

In [ ]:
# Each sample is a small field-realistic earth: folded, dipping thin beds,
# one to three normal faults (throw tapering out near the top), sometimes
# an angular unconformity with a draped overburden, imaged with a
# band-limited wavelet and band-limited noise at a random S/N.
# The labels stay pixel-exact because the geometry is ours by construction.

def ricker(f, dt, n):
    t = (np.arange(n) - n//2) * dt
    a = (np.pi * f * t) ** 2
    return (1 - 2*a) * np.exp(-a)

rng = np.random.default_rng(SEED)
w = ricker(F_HZ, 0.002, 61)
NPAD = NZ + 64                                       # pad kills wrap-around
wpad = np.zeros(NPAD); wpad[:61] = w
WF = np.fft.rfft(np.roll(wpad, -30), axis=0)         # fast convolution
zc = np.arange(NZ, dtype=float)                      # depth sample axis
xn = np.linspace(0.0, 1.0, NX)                       # normalized distance

X = np.zeros((N_SAMPLES, NZ, NX), dtype=np.float32)
Y = np.zeros((N_SAMPLES, NZ, NX), dtype=np.float32)

for i in range(N_SAMPLES):
    # --- structure: dip + an asymmetric fold --------------------------
    dip = rng.uniform(-28.0, 28.0)                   # samples across section
    x0 = rng.uniform(0.25, 0.75)
    amp = rng.uniform(0.0, 26.0) * rng.choice([0.0, 1.0], p=[0.25, 0.75])
    sl, sr = rng.uniform(0.10, 0.22, 2)
    sig = np.where(xn < x0, sl, sr)
    fold = amp * np.exp(-0.5 * ((xn - x0) / sig) ** 2)
    struct = dip * (xn - 0.5) - fold                 # relief (samples), (NX,)

    # --- thin beds ----------------------------------------------------
    n_beds = rng.integers(11, 18)
    tops = np.sort(rng.uniform(6, NZ + 18, n_beds))[:, None] + struct[None, :]

    # --- faults: 1..3, throw tapers to zero toward the surface --------
    n_f = rng.integers(1, 4)
    fxs, fsl, fth, fdie = (rng.uniform(0.18, 0.82, n_f),
                           rng.uniform(-0.35, 0.35, n_f),
                           rng.uniform(7, MAX_THROW + 1, n_f),
                           rng.uniform(6, NZ * 0.35, n_f))
    for j in range(n_f):
        xf = fxs[j] + fsl[j] * (tops - NZ/2) / NZ
        hw = 1.0 / (1.0 + np.exp(-(xn[None, :] - xf) / 0.006))
        taper = np.clip((tops - fdie[j]) / (NZ - fdie[j]), 0, 1.2)
        tops = tops + fth[j] * taper * hw

    # --- sometimes, an unconformity + draped overburden ---------------
    zu_used = None
    if rng.random() < 0.5:
        zu = rng.uniform(18, 52) + rng.uniform(-10, 10) * xn
        zu_used = zu
        tops = np.maximum(tops, zu[None, :])
        n_ov = rng.integers(2, 5)
        ov = np.sort(rng.uniform(4, zu.min() - 3, n_ov))[:, None] \
            + rng.uniform(0.1, 0.5) * (zu - zu.mean())[None, :]
        tops = np.vstack([ov, tops])

    tops = np.maximum.accumulate(np.sort(tops, axis=0), axis=0)

    # --- properties: compaction trend + alternation -------------------
    n_lay = tops.shape[0] + 1
    ai = (5200.0 + 28.0 * np.arange(n_lay)) \
        * np.where(np.arange(n_lay) % 2 == 0, 0.94, 1.07) \
        * (1.0 + 0.05 * rng.standard_normal(n_lay))
    idx = (zc[:, None, None] >= tops[None, :, :]).sum(axis=1)
    grid = ai[idx]

    rc = np.zeros_like(grid)
    rc[1:, :] = (grid[1:, :] - grid[:-1, :]) / (grid[1:, :] + grid[:-1, :])
    sec = np.fft.irfft(np.fft.rfft(rc, n=NPAD, axis=0) * WF[:, None], n=NPAD, axis=0)[:NZ]

    # band-limited noise at a random survey-quality S/N
    noise = rng.normal(0, 1, sec.shape)
    noise = np.fft.irfft(np.fft.rfft(noise, n=NPAD, axis=0) * WF[:, None], n=NPAD, axis=0)[:NZ]
    snr = rng.uniform(3.0, 8.0)
    noise *= np.sqrt((sec**2).mean()) / (snr * np.sqrt((noise**2).mean()) + 1e-12)
    X[i] = sec + noise

    # --- pixel-exact labels -------------------------------------------
    if LABEL == 'fault':
        for j in range(n_f):
            taper = np.clip((zc - fdie[j]) / (NZ - fdie[j]), 0, 1.2)
            xf = fxs[j] + fsl[j] * (zc - NZ/2) / NZ
            cols = np.clip((xf * (NX - 1)).astype(int), 0, NX - 1)
            on = fth[j] * taper > 1.5
            if zu_used is not None:
                on = on & (zc >= zu_used[cols])   # truncated above: no fault
            Y[i, zc[on].astype(int), cols[on]] = 1.0
    elif LABEL == 'horizon':
        for k in range(tops.shape[0]):
            cols = np.arange(NX)
            rows = np.clip(tops[k].astype(int), 0, NZ - 1)
            Y[i, rows, cols] = 1.0
    else:  # facies: the layer-index grid itself is the label
        Y[i] = idx

np.savez('ssm_training_pairs.npz', X=X, Y=Y)
print('saved', X.shape, '->', 'ssm_training_pairs.npz')

fig, ax = plt.subplots(2, 4, figsize=(10.5, 5.2))
for j in range(4):
    c = np.percentile(np.abs(X[j]), 99)
    ax[0, j].imshow(X[j], cmap='gray_r', vmin=-c, vmax=c, aspect='auto')
    ax[1, j].imshow(Y[j], cmap='hot', aspect='auto')
    ax[0, j].set_axis_off(); ax[1, j].set_axis_off()
ax[0, 0].set_title('section', loc='left')
ax[1, 0].set_title('label', loc='left')
plt.tight_layout(); plt.show()

## Where to go next

- The course: https://ogbonlab.com/textbooks/synthetic-seismic-modeling
- More notebooks: https://github.com/mosesfalowo/ogbonlab-notebooks
- Scale up: Devito, SPECFEM3D, Madagascar (course Part 12.3)